# Datos de Pulso TransMi en DataFrames

Este notebook consulta la API pública, descarga todas las páginas disponibles y las carga en `pandas.DataFrame`.

In [ ]:
# Si aún no instalaste el SDK desde la raíz del proyecto, ejecuta esta celda una vez:
# %pip install -e .

import pandas as pd
from pulso_transmi import PulsoTransmiClient


In [ ]:
API_URL = "https://pulso-transmi.72-60-245-2.sslip.io"
client = PulsoTransmiClient(base_url=API_URL, timeout=60)

meta = client.meta()
meta

In [ ]:
# Catálogo de estaciones
stations_df = client.stations()
print(f"Filas: {len(stations_df):,}")
display(stations_df.head())

## Geografía de las estaciones

Las coordenadas vienen en `latitude` y `longitude`. La siguiente gráfica muestra la distribución espacial de las estaciones en Bogotá.

In [ ]:
# Si matplotlib no está instalado, ejecuta: %pip install matplotlib
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 8))
for corridor, group in stations_df.groupby("corridor"):
    ax.scatter(
        group["longitude"],
        group["latitude"],
        s=90,
        alpha=0.85,
        label=corridor,
    )
    for _, station in group.iterrows():
        ax.annotate(
            station["station_name"],
            (station["longitude"], station["latitude"]),
            xytext=(5, 5),
            textcoords="offset points",
            fontsize=8,
        )

ax.set_title("Estaciones de Pulso TransMi")
ax.set_xlabel("Longitud")
ax.set_ylabel("Latitud")
ax.grid(True, alpha=0.3)
ax.legend(title="Corredor", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
# Observaciones: el SDK recorre automáticamente toda la paginación
observations_df = client.observations_dataframe(page_size=5000)
print(f"Filas: {len(observations_df):,}")
print(f"Rango: {observations_df['observed_at'].min()} — {observations_df['observed_at'].max()}")
display(observations_df.head())

In [ ]:
# Contexto (clima/eventos), también con paginación automática
context_df = client.context_dataframe(page_size=5000)
print(f"Filas: {len(context_df):,}")
display(context_df.head())

In [ ]:
# Ejemplo de filtro por estación y rango de fechas
station_07107_df = client.observations_dataframe(
    station_id="07107",
    start="2026-08-01T00:00:00-05:00",
    end="2026-08-07T23:59:59-05:00",
)
display(station_07107_df.head())

In [ ]:
# Opcional: guardar los DataFrames localmente
stations_df.to_csv("stations.csv", index=False)
observations_df.to_csv("observations.csv", index=False)
context_df.to_csv("context.csv", index=False)
print("Archivos guardados.")